# Data Exploration — AAPL 10-K 2023
Understand what raw SEC EDGAR filings look like before Phase 2 (parsing/chunking).

**Run order:** top to bottom. Requires `data/raw/AAPL/10-K_2023/primary.htm` to exist.
If not: `python scripts/download_filings.py --ticker AAPL --years 2023 --forms 10-K`

In [1]:
import re
import sys
import warnings

sys.path.insert(0, "..")

from pathlib import Path

from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from rich.console import Console
from rich.table import Table

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

FILING = Path("../data/raw/AAPL/10-K_2023/primary.htm")
assert FILING.exists(), f"Missing: {FILING} — run download_filings.py first"
print(f"File size: {FILING.stat().st_size:,} bytes ({FILING.stat().st_size / 1_048_576:.1f} MB)")

File size: 1,558,924 bytes (1.5 MB)


## 1. Document structure
EDGAR 10-K filings are **inline XBRL (iXBRL)** — not plain HTML.
Key implication: content lives in `<span>` tags with XBRL attributes, not `<p>` tags.
Phase 2 `html_parser.py` must handle this.

In [2]:
with open(FILING, "rb") as f:
    soup = BeautifulSoup(f, "lxml")

stats = {
    "tables": len(soup.find_all("table")),
    "spans": len(soup.find_all("span")),
    "divs": len(soup.find_all("div")),
    "paragraphs": len(soup.find_all("p")),  # expect ~0 (iXBRL, not plain HTML)
    "h1-h4": len(soup.find_all(["h1", "h2", "h3", "h4"])),  # expect ~0
}

all_text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
stats["words"] = len(all_text.split())
stats["chars"] = len(all_text)
stats["dollar_figures"] = len(re.findall(r"\$\s*[\d,]+", all_text))

console = Console()
t = Table(title="AAPL 10-K 2023 — Structure")
t.add_column("Element")
t.add_column("Count", justify="right")
for k, v in stats.items():
    t.add_row(k, f"{v:,}")
console.print(t)

print()
print("NOTE: paragraphs=0 and h1-h4=0 → this is iXBRL, not plain HTML.")
print("Content is in <span> tags. Phase 2 html_parser must extract spans, not <p>.")

 AAPL 10-K 2023 — Structure 
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Element        ┃   Count ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ tables         │      66 │
│ spans          │   3,389 │
│ divs           │   1,277 │
│ paragraphs     │       0 │
│ h1-h4          │       0 │
│ words          │  31,484 │
│ chars          │ 216,635 │
│ dollar_figures │     379 │
└────────────────┴─────────┘


NOTE: paragraphs=0 and h1-h4=0 → this is iXBRL, not plain HTML.
Content is in <span> tags. Phase 2 html_parser must extract spans, not <p>.


## 2. Sections in the document (10-K Items)
10-K structure: Part I (Items 1-4), Part II (Items 5-9A), Part III (Items 10-14), Part IV (Item 15).

In [3]:
items = re.findall(r"Item\s+\d+[A-Z]?\.\s+[A-Z][^\n]{5,60}", all_text)
# deduplicate (ToC + actual section both match)
seen = set()
unique_items = []
for item in items:
    key = item[:20]
    if key not in seen:
        seen.add(key)
        unique_items.append(item)

print("Sections found:")
for i in unique_items:
    print(" ", i.strip())

Sections found:
  Item 1. Business 1 Item 1A. Risk Factors 5 Item 1B. Unresolved Staff
  Item 1C. C ybersecurity 16 Item 2. Properties 17 Item 3. Legal Proceed
  Item 4. Mine Safety Disclosures 17 Part II Item 5. Market for Registr
  Item 7. Management’s Discussion and Analysis of Financial Condition a
  Item 7A. Quantitative and Qualitative Disclosures About Market Risk 26
  Item 8. Financial Statements and Supplementary Data 27 Item 9. Change
  Item 9A. Controls and Procedures 52 Item 9B. Other Information 53 Item
  Item 10. Directors, Executive Officers and Corporate Governance 53 Ite
  Item 14. Principal Accountant Fees and Services 53 Part IV Item 15. Ex
  Item 16. Form 10-K Summary 57 This Annual Report on Form 10-K (“Form 1
  Item 1. Business Company Background The Company designs, manufactures
  Item 1A. Risk Factors The Company’s business, reputation, results of o
  Item 1B. Unresolved Staff Comments None. Item 1C. Cybersecurity Not ap
  Item 2. Properties The Company’s headqu

## 3. Sample text from key sections
Check what actual content looks like for Item 1 (Business) and Item 7 (MD&A).

In [4]:
def show_section(label, pattern, chars=800):
    m = re.search(pattern, all_text)
    print(f"=== {label} ===")
    if m:
        print(m.group(0)[:chars])
    else:
        print("NOT FOUND")
    print()


# The ToC appears first — skip past it to find actual content
# Item 1 body content starts after 'Item 1A.' in second occurrence
occurrences = [m.start() for m in re.finditer(r"Item 1\. Business", all_text)]
print(f'"Item 1. Business" appears {len(occurrences)} times (1st=ToC, 2nd=actual section)')
print()

if len(occurrences) >= 2:
    body_start = occurrences[1]
    print("=== ITEM 1 BODY (first 800 chars) ===")
    print(all_text[body_start : body_start + 800])
    print()

# Net sales mentions
print("=== NET SALES MENTIONS ===")
for m in re.finditer(r"[Nn]et [Ss]ales[^.]{0,120}\.", all_text):
    print(" ", m.group(0)[:150])
    if len(list(re.finditer(r"[Nn]et [Ss]ales", all_text[: m.start()]))) > 8:
        break

"Item 1. Business" appears 2 times (1st=ToC, 2nd=actual section)

=== ITEM 1 BODY (first 800 chars) ===
Item 1. Business Company Background The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services. The Company’s fiscal year is the 52- or 53-week period that ends on the last Saturday of September. Products iPhone iPhone ® is the Company’s line of smartphones based on its iOS operating system. The iPhone line includes iPhone 15 Pro, iPhone 15, iPhone 14, iPhone 13 and iPhone SE ® . Mac Mac ® is the Company’s line of personal computers based on its macOS ® operating system. The Mac line includes laptops MacBook Air ® and MacBook Pro ® , as well as desktops iMac ® , Mac mini ® , Mac Studio ® and Mac Pro ® . iPad iPad ® is the Company’s line of multipurpose tablets based on its iPadOS ® operating sys

=== NET SALES MENTIONS ===
  net sales through its direct and indirect distribution channels a

## 4. Tables — the most important RAG challenge
Financial numbers (revenue, EPS, cash flow) live in tables. Plain `get_text()` breaks table structure.
Phase 2 must serialize tables to text (e.g. `Revenue: $383B (2023), $394B (2022)`).

In [5]:
tables = soup.find_all("table")
print(f"Total tables: {len(tables)}")
print()

# Show tables by size (text chars) — big tables = financial statements
table_sizes = [(i, len(t.get_text())) for i, t in enumerate(tables)]
table_sizes.sort(key=lambda x: x[1], reverse=True)

print("=== LARGEST TABLES (likely financial statements) ===")
for idx, size in table_sizes[:8]:
    preview = tables[idx].get_text(" | ", strip=True)[:80]
    print(f"Table {idx:2d}  {size:6,} chars  |  {preview}...")

print()
print("=== TABLE 20 FULL TEXT (shareholders equity) ===")
print(tables[20].get_text(" | ", strip=True)[:800])

Total tables: 66

=== LARGEST TABLES (likely financial statements) ===
Table 61   4,402 chars  |  Incorporated by Reference | Exhibit Number | Exhibit Description | Form | Exhibi...
Table 62   3,387 chars  |  Incorporated by Reference | Exhibit Number | Exhibit Description | Form | Exhibi...
Table 60   2,171 chars  |  Incorporated by Reference | Exhibit Number | Exhibit Description | Form | Exhibi...
Table 26   1,960 chars  |  Years ended | September 30, | 2023 | September 24, | 2022 | September 25, | 2021...
Table 24   1,343 chars  |  September 30, | 2023 | September 24, | 2022 | ASSETS: | Current assets: | Cash a...
Table 58   1,207 chars  |  How We Addressed the | Matter in Our Audit | We tested controls relating to the ...
Table 11   1,193 chars  |  Page | Part I | Item 1. | Business | 1 | Item 1A. | Risk Factors | 5 | Item 1B. ...
Table 25   1,081 chars  |  Years ended | September 30, | 2023 | September 24, | 2022 | September 25, | 2021...

=== TABLE 20 FULL TEXT (shareholders equ

## 5. Dollar figures — what numbers are in this document?

In [6]:
dollar_pat = r"\$\s*[\d,]+(?:\.\d+)?"
dollars = re.findall(dollar_pat, all_text)

# Parse to float, filter big numbers (millions)
big_numbers = []
for d in dollars:
    clean = d.replace("$", "").replace(",", "").strip()
    try:
        val = float(clean)
        if val > 1000:  # in millions
            big_numbers.append(val)
    except ValueError:
        pass

big_numbers.sort(reverse=True)
print(f"Total dollar figures: {len(dollars)}")
print(f"Figures > $1,000 (millions): {len(big_numbers)}")
print(f"Largest 10 values (in millions): {big_numbers[:10]}")
print()
print("These map to (billions): ", [f"${v / 1000:.1f}B" for v in big_numbers[:10]])

Total dollar figures: 379
Figures > $1,000 (millions): 230
Largest 10 values (in millions): [2591165000000.0, 394328.0, 394328.0, 394328.0, 394328.0, 383285.0, 383285.0, 383285.0, 383285.0, 365817.0]

These map to (billions):  ['$2591165000.0B', '$394.3B', '$394.3B', '$394.3B', '$394.3B', '$383.3B', '$383.3B', '$383.3B', '$383.3B', '$365.8B']


## 6. Chunking preview — what does splitting look like?
Rough estimate: how many 512-token chunks will this document produce?

In [7]:
# Rough token estimate: 1 token ≈ 4 chars (GPT/BERT rule of thumb)
CHARS_PER_TOKEN = 4
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
EFFECTIVE_CHUNK = CHUNK_SIZE - CHUNK_OVERLAP  # chars per stride

total_tokens_approx = len(all_text) / CHARS_PER_TOKEN
est_chunks = total_tokens_approx / EFFECTIVE_CHUNK

print(f"Total text chars:        {len(all_text):,}")
print(f"Approx tokens:           {int(total_tokens_approx):,}  (chars / 4)")
print(f"Chunk size (tokens):     {CHUNK_SIZE}")
print(f"Chunk overlap (tokens):  {CHUNK_OVERLAP}")
print(f"Est. chunks this doc:    ~{int(est_chunks)}")
print()
print("For 5 tickers x 4 years x (1 10-K + 3 10-Q) = 20 docs:")
print(f"  Est. total chunks:    ~{int(est_chunks * 20):,}  (rough, docs vary in size)")
print()
print("LEARN: this is why batch_size=32 in config — loading 50k+ chunks")
print("       into BGE-M3 in one shot would OOM. Batched embedding needed.")

Total text chars:        216,635
Approx tokens:           54,158  (chars / 4)
Chunk size (tokens):     512
Chunk overlap (tokens):  64
Est. chunks this doc:    ~120

For 5 tickers x 4 years x (1 10-K + 3 10-Q) = 20 docs:
  Est. total chunks:    ~2,417  (rough, docs vary in size)

LEARN: this is why batch_size=32 in config — loading 50k+ chunks
       into BGE-M3 in one shot would OOM. Batched embedding needed.


## 7. iXBRL structure — what Phase 2 parser must handle
This is the key challenge. Content is wrapped in XBRL-tagged spans like:
`<span name='us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax'>383,285</span>`

The XBRL tag names are actually useful metadata — they tell you what a number means semantically.
But for RAG we mostly just need clean text. Phase 2 strips XBRL and extracts text.

In [8]:
# Find XBRL-tagged spans (ix: namespace = inline XBRL)
xbrl_spans = soup.find_all(
    lambda tag: tag.name
    and "nonfraction" in tag.name.lower()
    or (tag.name == "span" and tag.get("name", "").startswith("us-gaap:"))
)

# Find financial concept spans
financial_spans = [tag for tag in soup.find_all(True) if tag.get("name", "").startswith("us-gaap:")]

print(f"Spans with us-gaap: XBRL concepts: {len(financial_spans)}")
print()
print("Sample XBRL-tagged financial values:")
for tag in financial_spans[:15]:
    concept = tag.get("name", "").replace("us-gaap:", "")
    value = tag.get_text(strip=True)
    if value and value.replace(",", "").replace(".", "").isdigit():
        print(f"  {concept:<50} = {value}")

print()
print("LEARN: XBRL concept names are structured taxonomy - Revenue, NetIncomeLoss,")
print("       EarningsPerShareBasic, etc. Could extract these as structured metadata")
print("       alongside free-text for better financial number retrieval.")

Spans with us-gaap: XBRL concepts: 1006

Sample XBRL-tagged financial values:
  RevenueRemainingPerformanceObligationPercentage    = 67
  RevenueRemainingPerformanceObligationPercentage    = 25
  RevenueRemainingPerformanceObligationPercentage    = 7
  RevenueRemainingPerformanceObligationPercentage    = 1

LEARN: XBRL concept names are structured taxonomy - Revenue, NetIncomeLoss,
       EarningsPerShareBasic, etc. Could extract these as structured metadata
       alongside free-text for better financial number retrieval.


## 8. Summary — what Phase 2 (html_parser.py) needs to do

Based on this exploration:

| Challenge | Observation | Phase 2 approach |
|---|---|---|
| No `<p>` tags | iXBRL uses `<span>` for text | Extract text from spans + divs |
| XBRL namespace garbage | First ~500 chars are metadata URIs | Skip `<head>` and XBRL header |
| 66 tables | Mix of cover page, ToC, financial stmts | Detect + serialize financial tables separately |
| Section structure | Items 1-15, found via regex | Split by Item number patterns |
| Dollar figures | 379 found, some malformed (`$ 2,591,165,000,000` = share count × price) | Filter by context |
| No H1-H4 headings | Structure implied by bold spans | Detect bold/larger spans as pseudo-headings |


## 9. Phase 1 finished — running the real `HTMLFilingParser`

Sections 1-8 above were exploration with raw regex/BeautifulSoup to understand the document.
That understanding is now implemented in `src/ingestion/html_parser.py` + `metadata_extractor.py`.
This section runs the actual parser and inspects its output (`ParsedFiling`).

In [9]:
import json
import time

from src.ingestion.html_parser import HTMLFilingParser
from src.ingestion.models import FilingRef

# Build the FilingRef from the metadata.json that download_filings.py already wrote —
# parse_filings.py does the same thing when scanning data/raw/ for the full pipeline.
raw_meta = json.loads(Path("../data/raw/AAPL/10-K_2023/metadata.json").read_text())
ref = FilingRef(
    ticker=raw_meta["ticker"],
    cik=raw_meta["cik"],
    form=raw_meta["form"],
    filing_date=raw_meta["filing_date"],
    report_date=raw_meta["report_date"],
    accession_no=raw_meta["accession_no"],
    primary_doc=raw_meta["primary_doc"],
)

t0 = time.time()
parsed = HTMLFilingParser().parse(FILING, ref)
print(f"Parsed in {time.time() - t0:.2f}s")

[06/19/26 15:31:35] INFO     Parsed AAPL 10-K 2023: 216,621 chars, 23 sections, 66 tables

Parsed in 0.19s


### 9a. Metadata — extracted from `dei:` XBRL tags, not regex on visible text

In [10]:
print("company_name:   ", parsed.company_name)
print("cik:            ", parsed.cik)
print("form:           ", parsed.form)
print("fiscal_year:    ", parsed.fiscal_year)
print("period_end_date:", parsed.period_end_date)
print("full_text chars:", f"{len(parsed.full_text):,}")

company_name:    Apple Inc.
cik:             0000320193
form:            10-K
fiscal_year:     2023
period_end_date: 2023-09-30
full_text chars: 216,621


### 9b. Sections — Item-numbered, ToC dropped

Compare to section 2 above: there, regex on raw text couldn't tell ToC entries apart from real
section headers. `html_parser.py` fixes this with a gap-based heuristic (see `LEARN` comment in
the source) — table of contents entries are packed within ~100 chars of each other; real section
bodies are separated by full paragraphs. Result: exactly the 23 canonical 10-K items, no ToC noise,
no duplicates.

In [11]:
sections_table = Table(title=f"{parsed.ticker} {parsed.form} {parsed.fiscal_year} — Sections")
sections_table.add_column("Item")
sections_table.add_column("Title (best-effort, first 8 words)")
sections_table.add_column("Chars", justify="right")

for s in parsed.sections:
    sections_table.add_row(s.item_label, s.title[:55], f"{len(s.text):,}")

console.print(sections_table)
print(f"\nTotal sections: {len(parsed.sections)} (expected 23 for a standard 10-K:")
print("1, 1A-1C, 2-5, 6, 7-7A, 8, 9-9C, 10-15, 16)")

                          AAPL 10-K 2023 — Sections                           
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Item    ┃ Title (best-effort, first 8 words)                      ┃  Chars ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Item 1  │ Business Company Background The Company designs, manufa │ 15,151 │
│ Item 1A │ Risk Factors The Company’s business, reputation, result │ 67,875 │
│ Item 1B │ Unresolved Staff Comments None.                         │     40 │
│ Item 1C │ Cybersecurity Not applicable. Apple Inc. | 2023 Form    │     71 │
│ Item 2  │ Properties The Company’s headquarters is located in Cup │    483 │
│ Item 3  │ Legal Proceedings Epic Games Epic Games, Inc. (“Epic”)  │  3,062 │
│ Item 4  │ Mine Safety Disclosures Not applicable. Apple Inc. |    │     88 │
│ Item 5  │ Market for Registrant’s Common Equity, Related Stockhol │  3,251 │
│ Item 6  │ [Reserved] Apple Inc. | 2023 Form 10-K |                │     51 │
│ Item 7  │ Management’s Discussion and Analysis of Financial Condi │ 15,488 │
│ Item 7A │ Quantitative and Qualitative Disclosures About Market R │  3,020 │
│ Item 8  │ Financial Statements and Supplementary Data Index to Co │ 62,262 │
│ Item 9  │ Changes in and Disagreements with Accountants on Accoun │     98 │
│ Item 9A │ Controls and Procedures Evaluation of Disclosure Contro │  4,499 │
│ Item 9B │ Other Information Insider Trading Arrangements On Augus │    802 │
│ Item 9C │ Disclosure Regarding Foreign Jurisdictions that Prevent │    101 │
│ Item 10 │ Directors, Executive Officers and Corporate Governance  │    401 │
│ Item 11 │ Executive Compensation The information required by this │    156 │
│ Item 12 │ Security Ownership of Certain Beneficial Owners and Man │    228 │
│ Item 13 │ Certain Relationships and Related Transactions, and Dir │    207 │
│ Item 14 │ Principal Accountant Fees and Services The information  │    213 │
│ Item 15 │ Exhibit and Financial Statement Schedules (a) Documents │ 12,287 │
│ Item 16 │ Form 10-K Summary None. Apple Inc. | 2023               │  2,137 │
└─────────┴─────────────────────────────────────────────────────────┴────────┘


Total sections: 23 (expected 23 for a standard 10-K:
1, 1A-1C, 2-5, 6, 7-7A, 8, 9-9C, 10-15, 16)


### 9c. Tables — kept raw, untouched, for Phase 2's `table_serializer.py`

In [12]:
print(f"raw_tables kept: {len(parsed.raw_tables)} (matches the 66 <table> tags found in section 4)")
print()
print("First 300 chars of table 24 (balance sheet, raw HTML — NOT flattened to text):")
print(parsed.raw_tables[24][:300])

raw_tables kept: 66 (matches the 66 <table> tags found in section 4)

First 300 chars of table 24 (balance sheet, raw HTML — NOT flattened to text):
<table style="border-collapse:collapse;display:inline-table;margin-bottom:5pt;vertical-align:text-bottom;width:100.000%"><tr><td style="width:1.0%"></td><td style="width:70.976%"></td><td style="width:0.1%"></td><td style="width:1.0%"></td><td style="width:12.496%"></td><td style="width:0.1%"></td><


### 9d. Persisting to `data/processed/` — the full CLI

`scripts/parse_filings.py` runs this same parser for every ticker/form/year in `configs/base.yaml`
(or CLI overrides), scanning `data/raw/` and writing `data/processed/{ticker}/{form}_{year}/parsed.json`
— idempotent, like `download_filings.py`. Already run for this filing above (see the file it wrote
below). To run for everything configured:

```bash
python scripts/parse_filings.py
```

In [13]:
processed_path = Path("../data/processed/AAPL/10-K_2023/parsed.json")
print(f"{processed_path}: {processed_path.stat().st_size:,} bytes")
print()
print("LEARN: this ParsedFiling JSON is the input contract for Phase 2 — chunkers.py,")
print("       cleaner.py, and table_serializer.py consume parsed.sections / parsed.raw_tables,")
print("       never the raw .htm file directly.")

../data/processed/AAPL/10-K_2023/parsed.json: 1,555,549 bytes

LEARN: this ParsedFiling JSON is the input contract for Phase 2 — chunkers.py,
       cleaner.py, and table_serializer.py consume parsed.sections / parsed.raw_tables,
       never the raw .htm file directly.
